# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
from typing import List

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")  # may be None for the public OpenAI API

assert OPENAI_API_KEY, "OPENAI_API_KEY is not set. Please populate your .env file."
assert TAVILY_API_KEY, "TAVILY_API_KEY is not set. Please populate your .env file."

In [4]:
# Connect to the persistent Chroma collection populated by Part 01.
embedding_kwargs = {
    "api_key": CHROMA_OPENAI_API_KEY,
    "model_name": "text-embedding-3-small",
}
if OPENAI_BASE_URL:
    embedding_kwargs["api_base"] = OPENAI_BASE_URL

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(**embedding_kwargs)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)
print(f"Connected to '{collection.name}' with {collection.count()} documents.")

# Reusable Tavily client
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

Connected to 'udaplay' with 15 documents.


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> List[dict]:
    """Semantic search: Finds the most relevant game entries in the local vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=5)
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]
    ids = results.get("ids", [[]])[0]

    output = []
    for doc_id, doc, meta, dist in zip(ids, documents, metadatas, distances):
        output.append({
            "id": doc_id,
            "Name": meta.get("Name"),
            "Platform": meta.get("Platform"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "document": doc,
            "distance": dist,
        })
    return output

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    """Structured judgement returned by the retrieval evaluator."""
    useful: bool = Field(
        ...,
        description="True only if the retrieved documents contain enough information to answer the user's question.",
    )
    description: str = Field(
        ...,
        description="Detailed explanation of the evaluation, citing which docs are useful or what is missing.",
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    system = SystemMessage(content=(
        "You are a strict retrieval-quality judge. Your task is to evaluate if the "
        "documents provided are enough to respond to the user's query. Give a detailed "
        "explanation, so it's possible to take an action to accept the documents or fall "
        "back to a web search. Only mark `useful=true` when the documents clearly contain "
        "the specific facts (game name, platform, year, etc.) needed to answer."
    ))
    user = UserMessage(content=(
        f"User question: {question}\n\n"
        f"Retrieved documents (JSON):\n{json.dumps(retrieved_docs, indent=2, default=str)}"
    ))
    response = judge.invoke([system, user], response_format=EvaluationReport)
    try:
        report = EvaluationReport.model_validate_json(response.content)
    except Exception:
        # Fallback: be conservative if structured parsing fails
        report = EvaluationReport(
            useful=False,
            description=f"Could not parse evaluator output: {response.content}",
        )
    return report.model_dump()

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> dict:
    """Web search: Finds information about a game on the public internet using Tavily.

    args:
    - question: a question about the game industry.

    Returns a dict with a short LLM-generated `answer` and a list of supporting `results`
    (each containing `title`, `url`, and `content`).
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
    )
    return {
        "answer": response.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content"),
            }
            for r in response.get("results", [])
        ],
    }

### Agent

In [8]:
INSTRUCTIONS = (
    "You are UdaPlay, an AI research assistant specialised in the video-game industry. "
    "Your job is to answer user questions about games (release dates, platforms, genres, "
    "publishers, descriptions) accurately and concisely.\n\n"
    "Workflow you MUST follow:\n"
    "1. ALWAYS call `retrieve_game` first to query the local knowledge base.\n"
    "2. Then call `evaluate_retrieval` with the same user question and the retrieved "
    "   documents to decide if those documents are enough.\n"
    "3. If the evaluator returns `useful=true`, answer using ONLY those documents.\n"
    "4. If the evaluator returns `useful=false`, call `game_web_search` to gather "
    "   missing information from the web, then craft your final answer.\n\n"
    "Final answer format:\n"
    "- Start with a direct, one-sentence answer.\n"
    "- Add a short paragraph with supporting details (game name, platform, year, genre, publisher).\n"
    "- Always cite your source: either `Source: internal knowledge base` or list the URLs from the web search.\n"
    "- If you genuinely cannot find the information, say so explicitly. Never invent facts."
)

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.2,
)

In [9]:
def ask(question: str, session_id: str = "demo") -> str:
    """Helper: invoke the agent and pretty-print the final assistant message."""
    print(f"\nQ: {question}")
    run = agent.invoke(question, session_id=session_id)
    final_state = run.get_final_state()
    final_msg = final_state["messages"][-1]
    answer = getattr(final_msg, "content", None) or ""
    print(f"A: {answer}\n" + "-" * 80)
    return answer

questions = [
    "When was Pok\u00e9mon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for q in questions:
    ask(q)


Q: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: Mortal Kombat X was not released for PlayStation 5; it was originally launched for PlayStation 4 in 2015.

While Mortal Kombat X is available as downloadable content for the PS5, it was not specifically released as a standalone title for that console. The game is known for its engaging gameplay and was developed by NetherRealm Studios, published by Warner Bros. Interactive Entertainment. 

Source: 
- [Mortal Kombat X - Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)
- [Mortal Kombat X - LaunchBox Games Database](https://gamesdb.launchbox-app.com/games/details/26542-mortal-kombat-x)
--------------------------------------------------------------------------------


### (Optional) Advanced

In [10]:
# The Agent class already maintains short-term memory per session_id (see
# `lib.memory.ShortTermMemory`) and is implemented on top of a state machine
# whose nodes are: message_prep -> llm_processor -> tool_executor -> llm_processor -> termination.
# Below we demonstrate multi-turn conversations using the same session_id and
# show how to inspect the stored runs.

ask("Who published Gran Turismo and on which platform did it debut?", session_id="chat-1")
ask("And what year was that?", session_id="chat-1")  # follow-up using stored context

runs = agent.get_session_runs("chat-1")
print(f"Total runs stored for session 'chat-1': {len(runs)}")


Q: Who published Gran Turismo and on which platform did it debut?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: Gran Turismo was published by Sony Computer Entertainment and debuted on the PlayStation 1 in 1997. 

This game is a realistic racing simulator that features a wide array of cars and tracks, setting a new standard for the racing genre. It was released in 1997 and is recognized for its significant impact on racing games. 

Source: internal knowledge base
--------------------------------------------------------------------------------

Q: And what year was that?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: Gran Turismo was released in the year 1997.

This iconic racing simulator debuted on the PlayStation 1 and was published by Sony Computer Entertainment, marking a significant milestone in the racing game genre. 

Source: internal knowledge base
--------------------------------------------------------------------------------
Total runs stored for session 'chat-1': 2
